In [12]:
import pandas as pd
import numpy as np
import os as o

print(o.getcwd())

D:\SAVING\Ecommerce_Dataset\Ecommerce_Dataset


In [5]:
customers = pd.read_csv("Customers.csv")
order_items = pd.read_csv("OrderItems.csv")
orders = pd.read_csv("Orders.csv")
payments = pd.read_csv("Payments.csv")
products = pd.read_csv("Products.csv")
promotions = pd.read_csv("Promotions.csv")
returns = pd.read_csv("Returns.csv")

In [3]:
BASE_PATH = r"D:\SAVING\Ecommerce_Dataset\Ecommerce_Dataset"

RAW_PATH = o.path.join(BASE_PATH, "Raw_Data")
CLEAN_PATH = o.path.join(BASE_PATH, "Cleaned_Data")
REPORT_PATH = o.path.join(BASE_PATH, "Reports")

o.makedirs(CLEAN_PATH, exist_ok=True)
o.makedirs(REPORT_PATH, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [6]:
tables = {
    "Customers": customers,
    "OrderItems": order_items,
    "Orders": orders,
    "Payments": payments,
    "Products": products,
    "Promotions": promotions,
    "Returns": returns,
}

for name, df in tables.items():
    print("=" * 50)
    print(f"Table: {name} | Shape: {df.shape}")
    print("Duplicates:", df.duplicated().sum())
    print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
    print("Columns & dtypes:\n", df.dtypes)

Table: Customers | Shape: (30000, 9)
Duplicates: 0
Missing values:
 Series([], dtype: int64)
Columns & dtypes:
 CustomerID           str
FirstName            str
LastName             str
Gender               str
Age                int64
City                 str
Country              str
SignupDate           str
CustomerSegment      str
dtype: object
Table: OrderItems | Shape: (300090, 7)
Duplicates: 0
Missing values:
 Series([], dtype: int64)
Columns & dtypes:
 OrderItemID            str
OrderID                str
ProductID              str
Quantity             int64
UnitPrice          float64
DiscountPercent    float64
LineTotal          float64
dtype: object
Table: Orders | Shape: (100000, 9)
Duplicates: 0
Missing values:
 PromotionID    30167
dtype: int64
Columns & dtypes:
 OrderID             str
CustomerID          str
OrderDate           str
OrderStatus         str
SalesChannel        str
ShippingFee     float64
TaxAmount       float64
OrderTotal      float64
PromotionID         s

In [10]:
# ==========================================
# ID INTEGRITY CHECK
# ==========================================

tables = datasets

checks = []

def check_unique_id(table_name, column_name):

    df = tables[table_name]

    duplicated_count = df[column_name].duplicated().sum()

    null_count = df[column_name].isnull().sum()

    checks.append({
        "Table": table_name,
        "Column": column_name,
        "Duplicate IDs": duplicated_count,
        "Null IDs": null_count,
        "Status": "PASS" if duplicated_count == 0 and null_count == 0 else "CHECK"
    })


# Primary Key Candidates

check_unique_id("Customers", "CustomerID")
check_unique_id("OrderItems", "OrderItemID")
check_unique_id("Orders", "OrderID")
check_unique_id("Payments", "PaymentID")
check_unique_id("Products", "ProductID")
check_unique_id("Promotions", "PromotionID")
check_unique_id("Returns", "ReturnID")


id_integrity_df = pd.DataFrame(checks)

display(id_integrity_df)

KeyError: 'Customers'

In [17]:
# ============================================================
# E-COMMERCE DATA AUDIT, CLEANING & VALIDATION PIPELINE
# ============================================================

import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("=" * 80)
print("E-COMMERCE DATA CLEANING PIPELINE")
print("=" * 80)


# ============================================================
# 1. PATHS
# ============================================================

BASE_PATH = r"D:\SAVING\Ecommerce_Dataset\Ecommerce_Dataset"

RAW_PATH = BASE_PATH
CLEAN_PATH = os.path.join(BASE_PATH, "Cleaned_Data")
REPORT_PATH = os.path.join(BASE_PATH, "Reports")

os.makedirs(CLEAN_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)


# ============================================================
# 2. FILES
# ============================================================

FILES = {
    "Customers": "Customers.csv",
    "OrderItems": "OrderItems.csv",
    "Orders": "Orders.csv",
    "Payments": "Payments.csv",
    "Products": "Products.csv",
    "Promotions": "Promotions.csv",
    "Returns": "Returns.csv"
}


# ============================================================
# 3. LOAD DATASETS
# ============================================================

datasets = {}

print("\n[1] LOADING DATASETS")
print("-" * 80)

for table_name, file_name in FILES.items():

    file_path = os.path.join(RAW_PATH, file_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path, low_memory=False)

    datasets[table_name] = df

    print(f"{table_name}: Loaded successfully")
    print(f"Shape: {df.shape}")


# ============================================================
# 4. INITIAL DATA AUDIT
# ============================================================

print("\n[2] INITIAL DATA AUDIT")
print("-" * 80)

audit_results = []

for table_name, df in datasets.items():

    audit_results.append({
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Memory_MB": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)
    })

initial_audit = pd.DataFrame(audit_results)

print(initial_audit)

initial_audit.to_csv(
    os.path.join(REPORT_PATH, "Initial_Audit.csv"),
    index=False
)


# ============================================================
# 5. MISSING VALUES REPORT
# ============================================================

print("\n[3] MISSING VALUES REPORT")
print("-" * 80)

missing_reports = []

for table_name, df in datasets.items():

    missing_count = df.isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100

    report = pd.DataFrame({
        "Table": table_name,
        "Column": df.columns,
        "Missing_Count": missing_count.values,
        "Missing_Percentage": missing_percentage.values
    })

    report = report[report["Missing_Count"] > 0]

    if not report.empty:
        missing_reports.append(report)

        print(f"\n{table_name}")
        print(report)

if missing_reports:

    missing_report = pd.concat(
        missing_reports,
        ignore_index=True
    )

    missing_report.to_csv(
        os.path.join(REPORT_PATH, "Missing_Values_Report.csv"),
        index=False
    )

else:

    print("No missing values found.")


# ============================================================
# 6. DATA TYPES REPORT
# ============================================================

print("\n[4] DATA TYPES REPORT")
print("-" * 80)

dtype_reports = []

for table_name, df in datasets.items():

    report = pd.DataFrame({
        "Table": table_name,
        "Column": df.columns,
        "Data_Type": df.dtypes.astype(str).values
    })

    dtype_reports.append(report)

dtype_report = pd.concat(
    dtype_reports,
    ignore_index=True
)

dtype_report.to_csv(
    os.path.join(REPORT_PATH, "Data_Types_Report.csv"),
    index=False
)

print(dtype_report)


# ============================================================
# 7. CREATE CLEANED COPIES
# ============================================================

print("\n[5] BASIC DATA CLEANING")
print("-" * 80)

cleaned_datasets = {}

for table_name, df in datasets.items():

    clean_df = df.copy()

    # Clean text columns
    for col in clean_df.select_dtypes(include=["object", "string"]).columns:

        clean_df[col] = (
            clean_df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

        clean_df[col] = clean_df[col].replace("", pd.NA)

    # Remove exact duplicate rows
    before_duplicates = len(clean_df)

    clean_df = clean_df.drop_duplicates()

    removed_duplicates = before_duplicates - len(clean_df)

    cleaned_datasets[table_name] = clean_df

    print(f"{table_name}")
    print(f"Rows before: {before_duplicates}")
    print(f"Duplicates removed: {removed_duplicates}")
    print(f"Rows after: {len(clean_df)}")
    print("-" * 50)


# ============================================================
# 8. PRIMARY KEY VALIDATION
# ============================================================

print("\n[6] PRIMARY KEY VALIDATION")
print("-" * 80)

primary_keys = {
    "Customers": "CustomerID",
    "OrderItems": "OrderItemID",
    "Orders": "OrderID",
    "Payments": "PaymentID",
    "Products": "ProductID",
    "Promotions": "PromotionID",
    "Returns": "ReturnID"
}

pk_reports = []

for table_name, pk_column in primary_keys.items():

    df = cleaned_datasets[table_name]

    if pk_column not in df.columns:

        print(f"{table_name}: Missing PK column {pk_column}")
        continue

    null_count = df[pk_column].isna().sum()

    duplicate_count = df[pk_column].duplicated().sum()

    report = {
        "Table": table_name,
        "Primary_Key": pk_column,
        "Null_PK": int(null_count),
        "Duplicate_PK": int(duplicate_count),
        "PK_Valid": null_count == 0 and duplicate_count == 0
    }

    pk_reports.append(report)

    print(f"\nTable: {table_name}")
    print(f"Primary Key: {pk_column}")
    print(f"Null IDs: {null_count}")
    print(f"Duplicate IDs: {duplicate_count}")
    print(f"Valid PK: {report['PK_Valid']}")


pk_report = pd.DataFrame(pk_reports)

pk_report.to_csv(
    os.path.join(REPORT_PATH, "Primary_Key_Report.csv"),
    index=False
)


# ============================================================
# 9. FOREIGN KEY VALIDATION
# ============================================================

print("\n[7] FOREIGN KEY VALIDATION")
print("-" * 80)

foreign_keys = [
    ("Orders", "CustomerID", "Customers", "CustomerID"),
    ("OrderItems", "OrderID", "Orders", "OrderID"),
    ("OrderItems", "ProductID", "Products", "ProductID"),
    ("Payments", "OrderID", "Orders", "OrderID"),
    ("Returns", "OrderID", "Orders", "OrderID"),
    ("Returns", "ProductID", "Products", "ProductID"),
    ("Orders", "PromotionID", "Promotions", "PromotionID")
]

fk_reports = []

for child_table, child_column, parent_table, parent_column in foreign_keys:

    child_df = cleaned_datasets[child_table]
    parent_df = cleaned_datasets[parent_table]

    child_values = child_df[child_column].dropna()
    parent_values = parent_df[parent_column].dropna()

    parent_set = set(parent_values)

    invalid_values = child_values[
        ~child_values.isin(parent_set)
    ]

    report = {
        "Child_Table": child_table,
        "Child_Column": child_column,
        "Parent_Table": parent_table,
        "Parent_Column": parent_column,
        "Invalid_Values": int(invalid_values.nunique()),
        "Invalid_Rows": int(len(invalid_values)),
        "FK_Valid": len(invalid_values) == 0
    }

    fk_reports.append(report)

    print(
        f"{child_table}.{child_column} -> "
        f"{parent_table}.{parent_column}"
    )

    print(f"Invalid rows: {len(invalid_values)}")
    print(f"Unique invalid values: {invalid_values.nunique()}")
    print("-" * 50)


fk_report = pd.DataFrame(fk_reports)

fk_report.to_csv(
    os.path.join(REPORT_PATH, "Foreign_Key_Report.csv"),
    index=False
)


# ============================================================
# 10. BUSINESS RULE VALIDATION
# ============================================================

print("\n[8] BUSINESS RULE VALIDATION")
print("-" * 80)

business_issues = []


def check_rule(table_name, column, condition, rule_name):

    df = cleaned_datasets[table_name]

    if column not in df.columns:
        return

    invalid_rows = df[condition(df[column])]

    business_issues.append({
        "Table": table_name,
        "Column": column,
        "Rule": rule_name,
        "Invalid_Rows": len(invalid_rows)
    })

    print(
        f"{table_name}.{column} | "
        f"{rule_name} | "
        f"Invalid rows: {len(invalid_rows)}"
    )


# Customers

check_rule(
    "Customers",
    "Age",
    lambda x: (x < 0) | (x > 120),
    "Age must be between 0 and 120"
)


# OrderItems

check_rule(
    "OrderItems",
    "Quantity",
    lambda x: x <= 0,
    "Quantity must be greater than zero"
)

check_rule(
    "OrderItems",
    "UnitPrice",
    lambda x: x < 0,
    "UnitPrice cannot be negative"
)

check_rule(
    "OrderItems",
    "DiscountPercent",
    lambda x: (x < 0) | (x > 100),
    "DiscountPercent must be between 0 and 100"
)

check_rule(
    "OrderItems",
    "LineTotal",
    lambda x: x < 0,
    "LineTotal cannot be negative"
)


# Orders

check_rule(
    "Orders",
    "ShippingFee",
    lambda x: x < 0,
    "ShippingFee cannot be negative"
)

check_rule(
    "Orders",
    "TaxAmount",
    lambda x: x < 0,
    "TaxAmount cannot be negative"
)

check_rule(
    "Orders",
    "OrderTotal",
    lambda x: x < 0,
    "OrderTotal cannot be negative"
)


# Products

check_rule(
    "Products",
    "UnitCost",
    lambda x: x < 0,
    "UnitCost cannot be negative"
)

check_rule(
    "Products",
    "UnitPrice",
    lambda x: x < 0,
    "UnitPrice cannot be negative"
)

check_rule(
    "Products",
    "StockQuantity",
    lambda x: x < 0,
    "StockQuantity cannot be negative"
)


# Payments

check_rule(
    "Payments",
    "AmountPaid",
    lambda x: x < 0,
    "AmountPaid cannot be negative"
)


# Returns

check_rule(
    "Returns",
    "ReturnQuantity",
    lambda x: x <= 0,
    "ReturnQuantity must be greater than zero"
)

check_rule(
    "Returns",
    "RefundAmount",
    lambda x: x < 0,
    "RefundAmount cannot be negative"
)


business_report = pd.DataFrame(business_issues)

business_report.to_csv(
    os.path.join(REPORT_PATH, "Business_Rules_Report.csv"),
    index=False
)


# ============================================================
# 11. DATE VALIDATION & CONVERSION
# ============================================================

print("\n[9] DATE VALIDATION")
print("-" * 80)

date_columns = {
    "Customers": ["SignupDate"],
    "Orders": ["OrderDate"],
    "Payments": ["PaymentDate"],
    "Promotions": ["StartDate", "EndDate"],
    "Returns": ["ReturnDate"]
}

date_reports = []

for table_name, columns in date_columns.items():

    df = cleaned_datasets[table_name]

    for column in columns:

        if column not in df.columns:
            continue

        original_values = df[column].copy()

        converted_dates = pd.to_datetime(
            original_values,
            errors="coerce"
        )

        invalid_dates = (
            original_values.notna() &
            converted_dates.isna()
        )

        invalid_count = invalid_dates.sum()

        cleaned_datasets[table_name][column] = converted_dates

        date_reports.append({
            "Table": table_name,
            "Column": column,
            "Invalid_Dates": int(invalid_count),
            "Min_Date": converted_dates.min(),
            "Max_Date": converted_dates.max()
        })

        print(f"\n{table_name}.{column}")
        print(f"Invalid dates: {invalid_count}")
        print(f"Min date: {converted_dates.min()}")
        print(f"Max date: {converted_dates.max()}")


date_report = pd.DataFrame(date_reports)

date_report.to_csv(
    os.path.join(REPORT_PATH, "Date_Validation_Report.csv"),
    index=False
)


# ============================================================
# 12. PROMOTION DATE VALIDATION
# ============================================================

print("\n[10] PROMOTION DATE VALIDATION")
print("-" * 80)

if (
    "StartDate" in cleaned_datasets["Promotions"].columns
    and "EndDate" in cleaned_datasets["Promotions"].columns
):

    promotions_df = cleaned_datasets["Promotions"]

    invalid_promotions = promotions_df[
        promotions_df["StartDate"] > promotions_df["EndDate"]
    ]

    print(f"Invalid promotions: {len(invalid_promotions)}")

    invalid_promotions.to_csv(
        os.path.join(REPORT_PATH, "Invalid_Promotions.csv"),
        index=False
    )


# ============================================================
# 13. LINE TOTAL VALIDATION
# ============================================================

print("\n[11] LINE TOTAL VALIDATION")
print("-" * 80)

order_items = cleaned_datasets["OrderItems"].copy()

required_columns = [
    "Quantity",
    "UnitPrice",
    "DiscountPercent",
    "LineTotal"
]

if all(col in order_items.columns for col in required_columns):

    # Assumes DiscountPercent is stored from 0 to 100

    expected_line_total = (
        order_items["Quantity"]
        * order_items["UnitPrice"]
        * (1 - order_items["DiscountPercent"] / 100)
    )

    line_total_difference = (
        order_items["LineTotal"] - expected_line_total
    ).abs()

    line_total_issues = order_items[
        line_total_difference > 0.01
    ].copy()

    line_total_issues["Expected_LineTotal"] = (
        expected_line_total[line_total_difference > 0.01]
    )

    line_total_issues["Difference"] = (
        line_total_difference[line_total_difference > 0.01]
    )

    print(f"LineTotal mismatches: {len(line_total_issues)}")

    line_total_issues.to_csv(
        os.path.join(REPORT_PATH, "LineTotal_Issues.csv"),
        index=False
    )


# ============================================================
# 14. ORDER TOTAL VALIDATION
# ============================================================

print("\n[12] ORDER TOTAL VALIDATION")
print("-" * 80)

order_items = cleaned_datasets["OrderItems"]
orders = cleaned_datasets["Orders"]

if (
    "OrderID" in order_items.columns
    and "LineTotal" in order_items.columns
    and "OrderID" in orders.columns
    and "OrderTotal" in orders.columns
):

    calculated_totals = (
        order_items.groupby("OrderID")["LineTotal"]
        .sum()
        .reset_index()
    )

    calculated_totals.rename(
        columns={"LineTotal": "Calculated_Items_Total"},
        inplace=True
    )

    order_comparison = orders.merge(
        calculated_totals,
        on="OrderID",
        how="left"
    )

    order_comparison["Calculated_Items_Total"] = (
        order_comparison["Calculated_Items_Total"].fillna(0)
    )

    order_comparison["Difference"] = (
        order_comparison["OrderTotal"]
        - order_comparison["Calculated_Items_Total"]
    )

    order_total_issues = order_comparison[
        order_comparison["Difference"].abs() > 0.01
    ]

    print(f"Order total differences: {len(order_total_issues)}")

    order_total_issues.to_csv(
        os.path.join(REPORT_PATH, "OrderTotal_Audit.csv"),
        index=False
    )


# ============================================================
# 15. SAVE CLEANED DATASETS
# ============================================================

print("\n[13] SAVING CLEANED DATASETS")
print("-" * 80)

for table_name, df in cleaned_datasets.items():

    output_path = os.path.join(
        CLEAN_PATH,
        f"{table_name}_Cleaned.csv"
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# ============================================================
# 16. FINAL AUDIT
# ============================================================

print("\n[14] FINAL AUDIT")
print("-" * 80)

final_audit_results = []

for table_name, df in cleaned_datasets.items():

    final_audit_results.append({
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        )
    })

final_audit = pd.DataFrame(final_audit_results)

print(final_audit)

final_audit.to_csv(
    os.path.join(REPORT_PATH, "Final_Audit.csv"),
    index=False
)


# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nTotal Tables:", len(cleaned_datasets))

total_rows = sum(
    len(df) for df in cleaned_datasets.values()
)

print("Total Records:", f"{total_rows:,}")

print("\nCleaned datasets saved in:")
print(CLEAN_PATH)

print("\nReports saved in:")
print(REPORT_PATH)

print("\nAvailable cleaned tables:")

for table_name in cleaned_datasets.keys():
    print(f"- {table_name}_Cleaned.csv")

print("\nNEXT STEP: SQL SERVER DATABASE CREATION & DATA IMPORT")
print("=" * 80)

E-COMMERCE DATA CLEANING PIPELINE

[1] LOADING DATASETS
--------------------------------------------------------------------------------
Customers: Loaded successfully
Shape: (30000, 9)
OrderItems: Loaded successfully
Shape: (300090, 7)
Orders: Loaded successfully
Shape: (100000, 9)
Payments: Loaded successfully
Shape: (100000, 6)
Products: Loaded successfully
Shape: (5000, 8)
Promotions: Loaded successfully
Shape: (20, 6)
Returns: Loaded successfully
Shape: (7587, 8)

[2] INITIAL DATA AUDIT
--------------------------------------------------------------------------------
        Table    Rows  Columns  Missing_Values  Duplicate_Rows  Memory_MB
0   Customers   30000        9               0               0      14.73
1  OrderItems  300090        7               0               0      64.96
2      Orders  100000        9           30167               0      38.53
3    Payments  100000        6               0               0      32.11
4    Products    5000        8               0      

In [15]:
import os

BASE_PATH = r"D:\SAVING\Ecommerce_Dataset\Ecommerce_Dataset"

print("Files found in folder:\n")

for file in os.listdir(BASE_PATH):
    print(file)

Files found in folder:

Audit.ipynb
Cleaned_Data
Customers.csv
OrderItems.csv
Orders.csv
Payments.csv
Products.csv
Promotions.csv
Reports
Returns.csv


In [18]:
# ============================================================
# FINAL DATA PREPARATION FOR SQL SERVER
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 80)
print("FINAL DATA PREPARATION FOR SQL")
print("=" * 80)


# ============================================================
# 1. PATHS
# ============================================================

BASE_PATH = r"D:\SAVING\Ecommerce_Dataset\Ecommerce_Dataset"

CLEAN_PATH = os.path.join(BASE_PATH, "Cleaned_Data")
FINAL_PATH = os.path.join(BASE_PATH, "SQL_Ready_Data")
REPORT_PATH = os.path.join(BASE_PATH, "Reports")

os.makedirs(FINAL_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)


# ============================================================
# 2. COPY CLEANED DATA
# ============================================================

sql_datasets = {}

for table_name, df in cleaned_datasets.items():

    sql_datasets[table_name] = df.copy()

print("\nDatasets copied successfully.")


# ============================================================
# 3. FIX OrderItems PRIMARY KEY
# ============================================================

print("\n[1] FIXING OrderItems PRIMARY KEY")

order_items = sql_datasets["OrderItems"].copy()

# Keep original ID for auditing
order_items["Original_OrderItemID"] = order_items["OrderItemID"].astype("string")

# Generate a new unique ID for every row
order_items["OrderItemID"] = [
    f"OI_{i:07d}"
    for i in range(1, len(order_items) + 1)
]

# Verify uniqueness
print("Rows:", len(order_items))
print("Unique OrderItemIDs:", order_items["OrderItemID"].nunique())
print("Null OrderItemIDs:", order_items["OrderItemID"].isna().sum())

sql_datasets["OrderItems"] = order_items


# ============================================================
# 4. STANDARDIZE ID COLUMNS
# ============================================================

print("\n[2] STANDARDIZING ID COLUMNS")

id_columns = {
    "Customers": ["CustomerID"],
    "OrderItems": ["OrderItemID", "Original_OrderItemID", "OrderID", "ProductID"],
    "Orders": ["OrderID", "CustomerID", "PromotionID"],
    "Payments": ["PaymentID", "OrderID"],
    "Products": ["ProductID"],
    "Promotions": ["PromotionID"],
    "Returns": ["ReturnID", "OrderID", "ProductID"]
}

for table_name, columns in id_columns.items():

    df = sql_datasets[table_name]

    for column in columns:

        if column in df.columns:

            df[column] = df[column].astype("string").str.strip()

    sql_datasets[table_name] = df


# ============================================================
# 5. STANDARDIZE DATE COLUMNS
# ============================================================

print("\n[3] STANDARDIZING DATE COLUMNS")

date_columns = {
    "Customers": ["SignupDate"],
    "Orders": ["OrderDate"],
    "Payments": ["PaymentDate"],
    "Promotions": ["StartDate", "EndDate"],
    "Returns": ["ReturnDate"]
}

for table_name, columns in date_columns.items():

    df = sql_datasets[table_name]

    for column in columns:

        if column in df.columns:

            df[column] = pd.to_datetime(
                df[column],
                errors="coerce"
            )

    sql_datasets[table_name] = df


# ============================================================
# 6. FINAL PRIMARY KEY VALIDATION
# ============================================================

print("\n[4] FINAL PRIMARY KEY VALIDATION")

primary_keys = {
    "Customers": "CustomerID",
    "OrderItems": "OrderItemID",
    "Orders": "OrderID",
    "Payments": "PaymentID",
    "Products": "ProductID",
    "Promotions": "PromotionID",
    "Returns": "ReturnID"
}

final_pk_results = []

for table_name, pk_column in primary_keys.items():

    df = sql_datasets[table_name]

    null_count = df[pk_column].isna().sum()
    duplicate_count = df[pk_column].duplicated().sum()

    valid = null_count == 0 and duplicate_count == 0

    final_pk_results.append({
        "Table": table_name,
        "PrimaryKey": pk_column,
        "Rows": len(df),
        "NullPK": int(null_count),
        "DuplicatePK": int(duplicate_count),
        "Valid": valid
    })

    print(
        f"{table_name}: "
        f"Null={null_count}, "
        f"Duplicates={duplicate_count}, "
        f"Valid={valid}"
    )

final_pk_report = pd.DataFrame(final_pk_results)

final_pk_report.to_csv(
    os.path.join(REPORT_PATH, "SQL_Final_PK_Validation.csv"),
    index=False
)


# ============================================================
# 7. FINAL FOREIGN KEY VALIDATION
# ============================================================

print("\n[5] FINAL FOREIGN KEY VALIDATION")

foreign_keys = [
    ("Orders", "CustomerID", "Customers", "CustomerID"),
    ("OrderItems", "OrderID", "Orders", "OrderID"),
    ("OrderItems", "ProductID", "Products", "ProductID"),
    ("Payments", "OrderID", "Orders", "OrderID"),
    ("Returns", "OrderID", "Orders", "OrderID"),
    ("Returns", "ProductID", "Products", "ProductID"),
    ("Orders", "PromotionID", "Promotions", "PromotionID")
]

final_fk_results = []

for child_table, child_column, parent_table, parent_column in foreign_keys:

    child_df = sql_datasets[child_table]
    parent_df = sql_datasets[parent_table]

    child_values = child_df[child_column].dropna()
    parent_values = parent_df[parent_column].dropna()

    invalid = child_values[
        ~child_values.isin(set(parent_values))
    ]

    final_fk_results.append({
        "ChildTable": child_table,
        "ChildColumn": child_column,
        "ParentTable": parent_table,
        "ParentColumn": parent_column,
        "InvalidRows": len(invalid),
        "Valid": len(invalid) == 0
    })

    print(
        f"{child_table}.{child_column} -> "
        f"{parent_table}.{parent_column}: "
        f"{len(invalid)} invalid"
    )

final_fk_report = pd.DataFrame(final_fk_results)

final_fk_report.to_csv(
    os.path.join(REPORT_PATH, "SQL_Final_FK_Validation.csv"),
    index=False
)


# ============================================================
# 8. FINAL DATASET AUDIT
# ============================================================

print("\n[6] FINAL DATASET AUDIT")

final_audit_results = []

for table_name, df in sql_datasets.items():

    final_audit_results.append({
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "MissingValues": int(df.isna().sum().sum()),
        "Duplicates": int(df.duplicated().sum()),
        "MemoryMB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        )
    })

final_sql_audit = pd.DataFrame(final_audit_results)

print(final_sql_audit)

final_sql_audit.to_csv(
    os.path.join(REPORT_PATH, "SQL_Final_Audit.csv"),
    index=False
)


# ============================================================
# 9. EXPORT SQL READY FILES
# ============================================================

print("\n[7] EXPORTING SQL READY DATASETS")

for table_name, df in sql_datasets.items():

    output_file = os.path.join(
        FINAL_PATH,
        f"{table_name}.csv"
    )

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Exported: {table_name}.csv")


# ============================================================
# 10. FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("FINAL PREPARATION COMPLETED")
print("=" * 80)

print("\nSQL Ready Folder:")
print(FINAL_PATH)

print("\nTables prepared:")

for table_name in sql_datasets.keys():
    print(f"- {table_name}")

print("\nTotal Records:")
print(sum(len(df) for df in sql_datasets.values()))

print("\nPrimary Keys Status:")
print(final_pk_report[["Table", "Valid"]])

print("\nForeign Keys Status:")
print(final_fk_report[["ChildTable", "ChildColumn", "Valid"]])

if final_pk_report["Valid"].all() and final_fk_report["Valid"].all():

    print("\nSUCCESS: All primary and foreign key checks passed.")

else:

    print("\nWARNING: Some validation checks failed. Review the reports.")

print("\nPython preparation stage is completed.")
print("NEXT STEP: SQL SERVER")

FINAL DATA PREPARATION FOR SQL

Datasets copied successfully.

[1] FIXING OrderItems PRIMARY KEY
Rows: 300090
Unique OrderItemIDs: 300090
Null OrderItemIDs: 0

[2] STANDARDIZING ID COLUMNS

[3] STANDARDIZING DATE COLUMNS

[4] FINAL PRIMARY KEY VALIDATION
Customers: Null=0, Duplicates=0, Valid=True
OrderItems: Null=0, Duplicates=0, Valid=True
Orders: Null=0, Duplicates=0, Valid=True
Payments: Null=0, Duplicates=0, Valid=True
Products: Null=0, Duplicates=0, Valid=True
Promotions: Null=0, Duplicates=0, Valid=True
Returns: Null=0, Duplicates=0, Valid=True

[5] FINAL FOREIGN KEY VALIDATION
Orders.CustomerID -> Customers.CustomerID: 0 invalid
OrderItems.OrderID -> Orders.OrderID: 0 invalid
OrderItems.ProductID -> Products.ProductID: 0 invalid
Payments.OrderID -> Orders.OrderID: 0 invalid
Returns.OrderID -> Orders.OrderID: 0 invalid
Returns.ProductID -> Products.ProductID: 0 invalid
Orders.PromotionID -> Promotions.PromotionID: 0 invalid

[6] FINAL DATASET AUDIT
        Table    Rows  Columns